In [0]:
from pyspark.sql import functions as F

CATALOG, SCHEMA = "workspace", "pitchside"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

SEASON_RE = r"^20[0-9]{2}-[0-9]{2}$"   # real seasons only, e.g. 2025-26
MIN_ROWS_PER_GW = 50                    # sparse gameweeks are excluded

KEYS = {
    "teams": "id",
    "players": "id",
    "fixtures": "id",
    "player_gw_stats": "id",
    "ml_predictions": "id",
}

def clean(name):
    df = spark.table(f"raw_{name}").drop("_loaded_at")
    before = df.count()
    df = df.filter(F.col("season").rlike(SEASON_RE))      # drop simulated seasons
    df = df.dropDuplicates([KEYS[name]])                  # one row per key
    return df, before

cleaned = {}
for name in KEYS:
    df, before = clean(name)
    cleaned[name] = df
    print(f"{name}: {before} raw -> {df.count()} after season filter and dedup")

# players: rename and convert units like the dbt staging model
players = cleaned["players"]
if "position" in players.columns:
    players = players.withColumnRenamed("position", "player_position")
if "cost_start" in players.columns:
    players = players.withColumn("start_price_millions", F.col("cost_start") / 10.0)
cleaned["players"] = players

# player_gw_stats: price in millions, then drop sparse gameweeks
stats = cleaned["player_gw_stats"]
if "value" in stats.columns:
    stats = stats.withColumn("price_millions", F.col("value") / 10.0)

gw_counts = stats.groupBy("season", "gameweek").count()
valid_gw = gw_counts.filter(F.col("count") >= MIN_ROWS_PER_GW).select("season", "gameweek")
stats = stats.join(valid_gw, ["season", "gameweek"], "inner")
cleaned["player_gw_stats"] = stats
print(f"player_gw_stats after sparse-gameweek filter: {stats.count()}")

for name, df in cleaned.items():
    df.write.mode("overwrite").format("delta").saveAsTable(f"cleaned_{name}")
    print(f"saved cleaned_{name}")

teams: 106 raw -> 40 after season filter and dedup
players: 3329 raw -> 1508 after season filter and dedup
fixtures: 498 raw -> 420 after season filter and dedup
player_gw_stats: 7505 raw -> 5811 after season filter and dedup
ml_predictions: 2112 raw -> 2112 after season filter and dedup
player_gw_stats after sparse-gameweek filter: 5809
saved cleaned_teams
saved cleaned_players
saved cleaned_fixtures
saved cleaned_player_gw_stats
saved cleaned_ml_predictions


In [0]:
for t in ["player_gw_stats", "fixtures", "teams", "players"]:
    print(t)
    spark.table(f"cleaned_{t}").printSchema()

player_gw_stats
root
 |-- season: string (nullable = true)
 |-- gameweek: long (nullable = true)
 |-- id: long (nullable = true)
 |-- player_id: long (nullable = true)
 |-- fixture_id: long (nullable = true)
 |-- is_live: boolean (nullable = true)
 |-- minutes: long (nullable = true)
 |-- goals_scored: long (nullable = true)
 |-- assists: long (nullable = true)
 |-- clean_sheets: long (nullable = true)
 |-- goals_conceded: long (nullable = true)
 |-- saves: long (nullable = true)
 |-- bonus: long (nullable = true)
 |-- bps: long (nullable = true)
 |-- yellow_cards: long (nullable = true)
 |-- red_cards: long (nullable = true)
 |-- own_goals: long (nullable = true)
 |-- penalties_saved: long (nullable = true)
 |-- penalties_missed: long (nullable = true)
 |-- ict_index: double (nullable = true)
 |-- influence: double (nullable = true)
 |-- creativity: double (nullable = true)
 |-- threat: double (nullable = true)
 |-- expected_goals: double (nullable = true)
 |-- expected_assists: doubl